In [0]:
# Tried to connect via DBFS mount but failed because DBFS mounts are not available on this workspace

storage_account_name = "retaildatalakemnt"
container_name = "medallion-project"
access_key= "xTIfBhjoWoI6aVdF5awujmzvMPruSiwt6QNAm4jxFuGVUoN6qsFVRrgSDO8RMppaBDUAGiOJ1hnk+ASt0lapgQ=="


mount_point = "/mnt/azure_adls"

dbutils.fs.mount(
    source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
    mount_point = mount_point,
    extra_configs = {
        f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": access_key
    }
)

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-4952640743555707>, line 10
      5 access_key= "xTIfBhjoWoI6aVdF5awujmzvMPruSiwt6QNAm4jxFuGVUoN6qsFVRrgSDO8RMppaBDUAGiOJ1hnk+ASt0lapgQ=="
      8 mount_point = "/mnt/azure_adls"
---> 10 dbutils.fs.mount(
     11     source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
     12     mount_point = mount_point,
     13     extra_configs = {
     14         f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": access_key
     15     }
     16 )

File /databricks/python_shell/lib/dbruntime/dbutils.py:172, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
    170 exc.__context__ = None
    171 exc.__cause__ = None
--> 172 raise exc

ExecutionError: An error occurred while calling o506.mount.
: shaded.databricks.org.apache.hadoop.fs.azure.AzureException: 

In [0]:
# mount point are now created using ABFSS 
storage_account_name = "retaildatalakemnt"
container_name = "medallion-project"
access_key = "XXXXXXXXXXXXXXXXXX=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    access_key
)

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/retail_lake"

In [0]:
# Creating paths for raw, bronze, silver and gold dataframes
raw_path    = f"{base_path}/raw"
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"
gold_path   = f"{base_path}/gold"

for p in [raw_path, bronze_path, silver_path, gold_path]:
    dbutils.fs.mkdirs(p)

In [0]:
dbutils.fs.mounts()

[MountInfo(mountPoint='/databricks-datasets', source='databricks-datasets', encryptionType=''),
 MountInfo(mountPoint='/Volumes', source='UnityCatalogVolumes', encryptionType=''),
 MountInfo(mountPoint='/databricks/mlflow-tracking', source='databricks/mlflow-tracking', encryptionType=''),
 MountInfo(mountPoint='/databricks/mlflow-registry', source='databricks/mlflow-registry', encryptionType=''),
 MountInfo(mountPoint='/Workspace', source='WorkspaceFiles', encryptionType='')]

In [0]:
dbutils.fs.ls("abfss://medallion-project@retaildatalakemnt.dfs.core.windows.net/")

[FileInfo(path='abfss://medallion-project@retaildatalakemnt.dfs.core.windows.net/retail_lake/', name='retail_lake/', size=0, modificationTime=1777730379000)]

In [0]:
# checking the file system to see if the files are present, and to know their path
dbutils.fs.ls("abfss://medallion-project@retaildatalakemnt.dfs.core.windows.net/retail_lake/raw")

[FileInfo(path='abfss://medallion-project@retaildatalakemnt.dfs.core.windows.net/retail_lake/raw/sales_data.csv', name='sales_data.csv', size=35624, modificationTime=1777731204000),
 FileInfo(path='abfss://medallion-project@retaildatalakemnt.dfs.core.windows.net/retail_lake/raw/store_data.csv', name='store_data.csv', size=523, modificationTime=1777731204000)]

In [0]:
# assigning the sale_df to the respective path
sale_df=spark.read.csv("abfss://medallion-project@retaildatalakemnt.dfs.core.windows.net/retail_lake/raw/sales_data.csv", header=True, inferSchema=True)

In [0]:
# assigning the store_df to the respective path
store_df=spark.read.csv("abfss://medallion-project@retaildatalakemnt.dfs.core.windows.net/retail_lake/raw/store_data.csv", header=True, inferSchema=True)

In [0]:
sale_df.createOrReplaceTempView("sales")
store_df.createOrReplaceTempView("store")

In [0]:
# from initial investigation we found that the store_size column has null values in the store_df,
#  so is it feasible to remove all those null values depends upon the toral amount of sales generated,
#  from the stores with store_size null in store_df. So before any operation we checked using SQL what are the total number of sales generated from the stores with store_size null.
# and it is found that it is not feasible to remove all those null values and filling them with "Unknown"/ avg can either break aggregation logic/ give biased results thus we are keeping it as it is as during aggregations nulls. "I preserved NULLs to avoid introducing bias, since Spark aggregations naturally handle NULLs. If needed, I would use statistical imputation along with a missing indicator column."

%sql SELECT st.store_id, round(sum(total_amount), 2) AS total_sales
FROM sales s
JOIN store st ON s.store_id = st.store_id
WHERE st.store_size IS NULL
GROUP BY st.store_id

store_id,total_sales
7.0,61380.04
11.0,72327.46
19.0,63595.56
17.0,65365.38
10.0,81501.49
13.0,78932.51
5.0,65479.82
16.0,44588.8


In [0]:
# import all major libraries used in this notebook
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from functools import *

In [0]:
# DATA CLEANING SALE_DF
display(sale_df)

sale_id,store_id,product_id,sale_date,quantity,total_amount
sale_1,19.0,46,2022-08-19,96,3024.78
sale_2,9.0,43,2021-03-15,100,4549.74
sale_3,3.0,20,2020-05-25,88,1477.8
sale_4,2.0,26,2022-06-25,30,231.85
sale_5,1.0,15,2021-06-15,83,2568.68
sale_6,14.0,3,2022-08-20,83,522.16
sale_7,8.0,14,2021-06-12,31,1301.42
sale_8,18.0,4,2021-12-01,95,680.5
sale_9,null,37,2022-01-18,78,1698.47
sale_10,15.0,45,2022-02-07,34,385.49


In [0]:
# checking number of nulls per column in sales_df
# pseudo code:
# 1. select all columns
# 2. for each column, count the number of nulls
# 3. cast the count to integer
# 4. alias the column with the column name
# 5. display the result

sale_df.select([sum(col(c).isNull().cast("integer")).alias(c) for c in sale_df.columns]).show()

+-------+--------+----------+---------+--------+------------+
|sale_id|store_id|product_id|sale_date|quantity|total_amount|
+-------+--------+----------+---------+--------+------------+
|     97|      47|         0|       94|       0|           0|
+-------+--------+----------+---------+--------+------------+



In [0]:
# checking rows with null values
# pseudo code:
# 1. for each column, check if the column is null
# 2. reduce the list of booleans to a single boolean by ORing them
# 3. filter the rows where the boolean is true
# 4. display the result

sale_df.filter(reduce(lambda x,y:x|y,(col(c).isNull()for c in sale_df.columns))).display()

sale_id,store_id,product_id,sale_date,quantity,total_amount
sale_9,null,37,2022-01-18,78,1698.47
sale_12,1.0,35,null,56,2017.73
null,2.0,16,2022-01-29,70,3230.59
null,5.0,12,2021-01-28,61,1075.45
null,1.0,15,2020-10-27,28,1309.62
sale_23,2.0,48,null,84,1076.87
sale_26,14.0,1,null,9,148.27
sale_35,null,34,2022-03-11,98,2600.47
sale_36,4.0,17,null,73,2985.87
sale_38,9.0,42,null,56,1668.19


In [0]:
# filling NULL values in quantity and total amount with 0
sale_df=sale_df.fillna({"quantity":0,"total_amount":0})

In [0]:
# dropping duplicates from sale df
sale_df=sale_df.dropDuplicates()

In [0]:
# storing only those values in the dataframe which have not NULL values in the columns sale_id, store_id and sale_date
sale_df = sale_df.filter(
    col("sale_id").isNotNull() &
    col("store_id").isNotNull() &
    col("sale_date").isNotNull()
)

In [0]:
# keeping only those values in the dataframe which have quantity > 0 and total amount > 0
sale_df=sale_df.filter((col("quantity")>0) & (col("total_amount")>0))

In [0]:
sale_df.count()

734

In [0]:
# recheck again to see any column with null values

sale_df.select([sum(col(c).isNull().cast("integer")).alias(c) for c in sale_df.columns]).show()

+-------+--------+----------+---------+--------+------------+
|sale_id|store_id|product_id|sale_date|quantity|total_amount|
+-------+--------+----------+---------+--------+------------+
|      0|       0|         0|        0|       0|           0|
+-------+--------+----------+---------+--------+------------+



In [0]:
# recheck again to see any row with null values

sale_df.filter(reduce(lambda x,y:x|y,(col(c).isNull()for c in sale_df.columns))).show()

+-------+--------+----------+---------+--------+------------+
|sale_id|store_id|product_id|sale_date|quantity|total_amount|
+-------+--------+----------+---------+--------+------------+
+-------+--------+----------+---------+--------+------------+



In [0]:
# saving cleaned sale_df to bronze medallion
sales_bronze_path = f"{bronze_path}/sales"

sale_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(sales_bronze_path)

In [0]:
# registers an existing Delta dataset in the metastore so it can be queried using SQL.
# separates storage and metadata by creating an external table pointing to data in ADLS, enabling scalable and decoupled data access.
# creates table name "retail_lakehouse.bronze_sales"
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_lakehouse")

sale_df.write.format("delta").mode("overwrite").saveAsTable("retail_lakehouse.bronze_sales")

In [0]:
%sql
select * from retail_lakehouse.bronze_sales

sale_id,store_id,product_id,sale_date,quantity,total_amount
sale_420,5.0,6,2022-07-16,37,295.1
sale_755,4.0,49,2021-05-04,12,505.45
sale_576,20.0,33,2021-02-07,2,89.61
sale_845,19.0,24,2020-06-16,97,2289.33
sale_999,18.0,11,2020-03-08,63,810.25
sale_60,8.0,32,2021-12-19,63,1816.5
sale_364,5.0,5,2021-10-02,63,2923.55
sale_17,6.0,1,2022-02-28,25,1047.87
sale_228,11.0,37,2021-11-27,37,394.83
sale_717,5.0,37,2020-02-05,66,2603.03


In [0]:
# DATA CLEANING STORE_DF
store_df.display()

store_id,store_region,store_size,open_date
1.0,West,2263.0,null
2.0,West,1510.0,2009-07-07
3.0,West,4989.0,2000-08-23
4.0,West,4942.0,2016-08-12
5.0,North,null,2002-06-09
null,East,3107.0,2007-03-28
7.0,North,null,2005-07-19
8.0,West,691.0,null
9.0,North,3653.0,2018-12-24
10.0,East,null,2006-01-18


In [0]:
# checking number of NULL values in each column
store_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in store_df.columns]).show()

+--------+------------+----------+---------+
|store_id|store_region|store_size|open_date|
+--------+------------+----------+---------+
|       2|           0|         8|        2|
+--------+------------+----------+---------+



In [0]:
store_df.filter(reduce(lambda x,y:x|y,(col(c).isNull()for c in store_df.columns))).show()

+--------+------------+----------+----------+
|store_id|store_region|store_size| open_date|
+--------+------------+----------+----------+
|     1.0|        West|    2263.0|      NULL|
|     5.0|       North|      NULL|2002-06-09|
|    NULL|        East|    3107.0|2007-03-28|
|     7.0|       North|      NULL|2005-07-19|
|     8.0|        West|     691.0|      NULL|
|    10.0|        East|      NULL|2006-01-18|
|    11.0|        East|      NULL|2018-04-22|
|    13.0|        West|      NULL|2009-10-03|
|    NULL|        West|    1968.0|2001-04-08|
|    16.0|        East|      NULL|2003-10-01|
|    17.0|        East|      NULL|2010-09-23|
|    19.0|       North|      NULL|2018-10-12|
+--------+------------+----------+----------+



In [0]:
# storing only those values in the store_df dataframe which do not have NULL values in the columns store_id
# Since store_id is a primary key, missing values would break referential integrity, 
# so I filtered them out. For non-key attributes, I preserved the data to avoid unnecessary data loss.
store_df=store_df.filter(
    col("store_id").isNotNull()
)

In [0]:
# recheck NULL values in each column and preserving nulls in store size column
# keeping nulls in store_size column as per the above discussion. 
store_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in store_df.columns]).show()

+--------+------------+----------+---------+
|store_id|store_region|store_size|open_date|
+--------+------------+----------+---------+
|       0|           0|         8|        2|
+--------+------------+----------+---------+



In [0]:
store_df.display()

store_id,store_region,store_size,open_date
1.0,West,2263.0,null
2.0,West,1510.0,2009-07-07
3.0,West,4989.0,2000-08-23
4.0,West,4942.0,2016-08-12
5.0,North,null,2002-06-09
7.0,North,null,2005-07-19
8.0,West,691.0,null
9.0,North,3653.0,2018-12-24
10.0,East,null,2006-01-18
11.0,East,null,2018-04-22


In [0]:
# saving cleaned sale_df to bronze medallion
store_bronze_path = f"{bronze_path}/store"

store_df.write.format("delta").mode("overwrite").save(store_bronze_path)

In [0]:
store_df.count()

18

In [0]:
# COMBINED DATA -- SILVER LAYER
combined_df = sale_df.join(store_df, on="store_id", how="inner").orderBy(col("store_id").asc())
display(combined_df)

store_id,sale_id,product_id,sale_date,quantity,total_amount,store_region,store_size,open_date
1.0,sale_734,46,2021-05-29,24,1041.99,West,2263.0,null
1.0,sale_748,8,2020-05-22,20,229.62,West,2263.0,null
1.0,sale_305,40,2022-12-01,77,2943.48,West,2263.0,null
1.0,sale_5,15,2021-06-15,83,2568.68,West,2263.0,null
1.0,sale_54,16,2022-02-28,99,3016.89,West,2263.0,null
1.0,sale_696,48,2020-09-27,21,439.31,West,2263.0,null
1.0,sale_468,38,2020-12-31,87,947.75,West,2263.0,null
1.0,sale_637,9,2021-07-31,90,3510.25,West,2263.0,null
1.0,sale_668,4,2020-06-29,62,1864.1,West,2263.0,null
1.0,sale_887,26,2021-10-09,66,1366.52,West,2263.0,null


In [0]:
# saving combined df to silver layer of medallion architecture

combined_df.write.format("delta").mode("overwrite").save(silver_path + "/combined_sales")

In [0]:
# external location is not configured so I created managed Delta tables for SQL querying and kept the ADLS medallion folders for pipeline storage.
# saving combined_df with table name "retail_lakehouse.silver_combined_sales"
combined_df.write.format("delta").mode("overwrite").saveAsTable("retail_lakehouse.silver_combined_sales")

In [0]:
# AGGREGATED DATA -- GOLD LAYER
# Sum of sales per store grouped by store_id from sale_df 
sales_per_store = sale_df.groupBy("store_id").agg(round(sum("total_amount"),2).alias("total_sales"))
sales_per_store.display()

store_id,total_sales
8.0,67746.38
7.0,52866.74
18.0,32691.26
1.0,41749.67
4.0,38284.52
11.0,66396.06
14.0,41333.42
19.0,49990.34
3.0,40979.65
2.0,47392.2


Databricks visualization. Run in Databricks to view.

In [0]:
# saving sales_per_store to gold layer of medallion architecture
sales_per_store.write.format("delta").mode("overwrite").save(gold_path + "/sales_per_store")

In [0]:
sales_per_store1 = combined_df.groupBy("store_id").agg(round(sum("total_amount"),2).alias("total_sales"))
sales_per_store1.display()

store_id,total_sales
1.0,41749.67
2.0,47392.2
3.0,40979.65
4.0,38284.52
5.0,49333.47
7.0,52866.74
8.0,67746.38
9.0,49838.1
10.0,72579.89
11.0,66396.06


Databricks visualization. Run in Databricks to view.

In [0]:
#Sales per region

sales_per_region = combined_df.groupBy("store_region").agg(round(sum("total_amount"),2).alias("total_sales"))
sales_per_region.display()
# saving sales_per_region to gold layer of medallion architecture
sales_per_region.write.format("delta").mode("overwrite").save(gold_path + "/sales_per_region")

store_region,total_sales
East,233534.13
West,369522.38
North,296419.3


Databricks visualization. Run in Databricks to view.

In [0]:
# checking which columns do have null values in store_size column as per combined_df
combined_df.filter(col("store_size").isNull()).select("store_id").distinct().show()

+--------+
|store_id|
+--------+
|     7.0|
|    11.0|
|    19.0|
|    17.0|
|    10.0|
|    13.0|
|     5.0|
|    16.0|
+--------+



In [0]:
# Sales per sq.feet 
sales_per_sqft = (
    combined_df
    .filter(col("store_size").isNotNull())
    .groupBy("store_id", "store_size")
    .agg(round(sum("total_amount"), 2).alias("total_sales_store"))
    .withColumn("sales_per_sqft", round(col("total_sales_store") / col("store_size"), 2))
)

display(sales_per_sqft)

store_id,store_size,total_sales_store,sales_per_sqft
1.0,2263.0,41749.67,18.45
2.0,1510.0,47392.2,31.39
3.0,4989.0,40979.65,8.21
4.0,4942.0,38284.52,7.75
8.0,691.0,67746.38,98.04
9.0,3653.0,49838.1,13.64
12.0,840.0,34734.9,41.35
14.0,1116.0,41333.42,37.04
18.0,1191.0,32691.26,27.45
20.0,4867.0,53057.23,10.9


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# saving sales_per_sqft to gold layer of medallion architecture
sales_per_sqft.write.format("delta").mode("overwrite").save(gold_path + "/sales_per_sqft")

In [0]:
## two SQL tables created for querying
## retail_lakehouse.bronze_sales
## retail_lakehouse.silver_combined_sales

In [0]:
%sql

SELECT * FROM retail_lakehouse.bronze_sales LIMIT 5

sale_id,store_id,product_id,sale_date,quantity,total_amount
sale_420,5.0,6,2022-07-16,37,295.1
sale_755,4.0,49,2021-05-04,12,505.45
sale_576,20.0,33,2021-02-07,2,89.61
sale_845,19.0,24,2020-06-16,97,2289.33
sale_999,18.0,11,2020-03-08,63,810.25


In [0]:
# product wise revenue
sales_per_product=spark.sql("""SELECT product_id, round(sum(total_amount),2) AS total_sales FROM retail_lakehouse.bronze_sales GROUP BY product_id ORDER BY product_id""") 
sales_per_product.display()

product_id,total_sales
1,17606.97
2,19817.89
3,16622.69
4,14846.08
5,20805.75
6,19062.15
7,14140.86
8,27097.7
9,14489.5
10,12535.95


Databricks visualization. Run in Databricks to view.

In [0]:
# saving sales_per_product to gold layer of medallion architecture
sales_per_product.write.format("delta").mode("overwrite").save(gold_path + "/product_sales")

In [0]:
# top 10 products generating maximum revenue

top10sales_product = spark.sql("""
SELECT *
FROM (
    SELECT 
        product_id,
        ROUND(SUM(total_amount),2) AS sales,
        ROW_NUMBER() OVER (ORDER BY SUM(total_amount) DESC) AS rank
    FROM retail_lakehouse.bronze_sales
    GROUP BY product_id
) t
WHERE rank <= 10
""")

display(top10sales_product)

product_id,sales,rank
14,36981.83,1
15,34156.95,2
46,31484.27,3
18,29982.79,4
32,29834.7,5
11,29262.31,6
12,27879.9,7
23,27782.83,8
8,27097.7,9
28,26523.82,10


Databricks visualization. Run in Databricks to view.

In [0]:
top10sales_product.write.format("delta").mode("overwrite").save(gold_path + "/top10sales_product")

In [0]:
# top 10 selling products quantity wise

top10sales_product_quantity = spark.sql("""
SELECT *
FROM (
    SELECT 
        product_id,
        SUM(quantity) AS quantity_sold,
        ROW_NUMBER() OVER (ORDER BY SUM(quantity) DESC) AS rank
    FROM retail_lakehouse.bronze_sales
    GROUP BY product_id
) t
WHERE rank <= 10
""")

display(top10sales_product_quantity)

product_id,quantity_sold,rank
14,1262,1
15,1202,2
22,1186,3
30,1180,4
32,1025,5
18,1023,6
8,983,7
26,981,8
12,978,9
5,946,10


Databricks visualization. Run in Databricks to view.

In [0]:
top10sales_product_quantity.write.format("delta").mode("overwrite").save(gold_path + "/top10sales_product_quantity")

In [0]:

#

goods_sold_per_region=spark.sql("""
    SELECT
        store_id,
        store_region,
        SUM(quantity) AS total_quantity_sold
    FROM retail_lakehouse.silver_combined_sales
    GROUP BY store_id, store_region
""")

display(goods_sold_per_region)

store_id,store_region,total_quantity_sold
1.0,West,1556
2.0,West,1648
4.0,West,1240
11.0,East,2178
14.0,North,1778
13.0,West,2351
5.0,North,1818
8.0,West,2404
20.0,North,2138
16.0,East,1554


Databricks visualization. Run in Databricks to view.

In [0]:
goods_sold_per_region.write.format("delta").mode("overwrite").save(gold_path + "/goods_sold_per_region")

In [0]:
## YEAR ON YEAR ROLLING PERCENTAGE
## 2023 REPRESENTS PARTIAL YEAR DATA

revenue_yoy = spark.sql("""
    SELECT
        year,
        total_revenue,
        ROUND(
            (total_revenue - LAG(total_revenue) OVER (ORDER BY year))
            * 100.0
            / NULLIF(LAG(total_revenue) OVER (ORDER BY year), 0),
        2) AS yoy_percentage
    FROM (
        SELECT
            YEAR(sale_date) AS year,
            round(SUM(total_amount),2) AS total_revenue
        FROM retail_lakehouse.silver_combined_sales
        GROUP BY YEAR(sale_date)
    ) t
    ORDER BY year
""")

display(revenue_yoy)

year,total_revenue,yoy_percentage
2020,304396.29,null
2021,301965.04,-0.8
2022,292422.06,-3.16
2023,692.42,-99.76


Databricks visualization. Run in Databricks to view.

In [0]:
revenue_yoy.write.format("delta").mode("overwrite").save(gold_path + "/revenue_yoy")